<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/colab/blob/main/notebooks/05_gpu_monte_carlo_finance.ipynb)


# colab — GPU Monte Carlo for Quantitative Finance

## _Option Pricing and Greeks at GPU Speed on a Free T4_

**&copy; Dr. Yves J. Hilpisch**<br>AI-Powered by Different LLMs.

## How to Use This Notebook

- **Goal**: Price European and barrier options via Monte Carlo simulation
  and compare CPU (NumPy) vs GPU (PyTorch CUDA) performance.
- **Hardware**: Designed for Google Colab (T4 default). GPU is auto-detected.
- **Data**: Fully synthetic GBM paths — no market data download required.
- **GPU Reference**: See `notebooks/colab_gpus.md` for the GPU comparison.

### Roadmap

1. **Environment**: Check GPU and import NumPy, PyTorch, Matplotlib.
2. **Parameters**: Define Black-Scholes-Merton parameters.
3. **GBM Paths**: Simulate geometric Brownian motion — CPU vs GPU.
4. **European Call**: Price a vanilla call option from the terminal prices.
5. **Barrier Option**: Price an up-and-out call with path dependency.
6. **Greeks**: Estimate Delta via finite differences on the GPU.
7. **Summary**: Side-by-side timing table.
8. **Optional**: Estimate historical volatility from SPY via `yfinance`.

### Environment Setup

We verify the GPU and install the minimal set of packages needed for
Monte Carlo simulation and plotting.

In [ ]:
#@title Check GPU and install packages
!nvidia-smi
!pip -q install matplotlib numpy torch

In [ ]:
#@title Imports and helpers
import time
import numpy as np
import torch
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch device: {device}")
print(f"PyTorch version: {torch.__version__}")

def bench(label, fn, *args, **kwargs):
    start = time.perf_counter()
    result = fn(*args, **kwargs)
    elapsed = time.perf_counter() - start
    print(f"{label}: {elapsed:.3f} s")
    return result, elapsed

### Market Parameters

Standard Black-Scholes-Merton assumptions for a European call option.
All values are synthetic and configurable.

In [ ]:
#@title BSM parameters
S0 = 100.0      # initial spot price
K = 100.0       # strike price
T = 1.0         # time to maturity (years)
r = 0.05        # risk-free rate
sigma = 0.2     # volatility
n_paths = 2_000_000  # number of Monte Carlo paths
n_steps = 252   # trading days per year
dt = T / n_steps
seed = 42

print(f"Paths: {n_paths:,}, Steps: {n_steps}, dt: {dt:.6f}")

### Geometric Brownian Motion Paths

Simulate terminal stock prices under GBM.
The GPU can generate and reduce millions of paths in parallel.

In [ ]:
#@title CPU path simulation (NumPy)
np.random.seed(seed)

def gbm_cpu(n_paths, n_steps, S0, r, sigma, dt):
    z = np.random.standard_normal((n_steps, n_paths))
    log_returns = (r - 0.5 * sigma ** 2) * dt + sigma * np.sqrt(dt) * z
    path_sum = log_returns.sum(axis=0)
    S_T = S0 * np.exp(path_sum)
    return S_T

S_T_cpu, t_cpu_gbm = bench("CPU GBM", gbm_cpu,
                          n_paths, n_steps, S0, r, sigma, dt)

In [ ]:
#@title GPU path simulation (PyTorch CUDA)
torch.manual_seed(seed)

def gbm_gpu(n_paths, n_steps, S0, r, sigma, dt, device):
    z = torch.randn((n_steps, n_paths), device=device)
    log_returns = (r - 0.5 * sigma ** 2) * dt \
        + sigma * np.sqrt(dt) * z
    path_sum = log_returns.sum(dim=0)
    S_T = S0 * torch.exp(path_sum)
    return S_T

S_T_gpu, t_gpu_gbm = bench("GPU GBM", gbm_gpu,
                           n_paths, n_steps, S0, r, sigma, dt, device)
torch.cuda.synchronize() if device == "cuda" else None

speedup_gbm = t_cpu_gbm / t_gpu_gbm
print(f"\n>>> GBM speed-up: {speedup_gbm:.1f}×")

### European Call Option Pricing

A vanilla European call is priced by averaging discounted payoffs over
the terminal stock prices already generated above. The payoff computation
is so lightweight that its runtime is negligible next to path generation;
we therefore show the price only, not a separate benchmark.

In [ ]:
#@title European call price (from terminal prices)
# CPU
payoff_cpu = np.maximum(S_T_cpu - K, 0.0)
price_cpu = np.exp(-r * T) * payoff_cpu.mean()

# GPU
payoff_gpu = torch.maximum(
    S_T_gpu - K,
    torch.tensor(0.0, device=S_T_gpu.device),
)
price_gpu = (
    torch.exp(torch.tensor(-r * T, device=S_T_gpu.device))
    * payoff_gpu.mean()
).item()
torch.cuda.synchronize() if device == "cuda" else None

print(f"CPU price: {price_cpu:.4f}")
print(f"GPU price: {price_gpu:.4f}")

### Barrier Option Pricing

An up-and-out call: the payoff is zero if the spot ever breaches a
barrier level during the life of the option.
This requires the full path (not just terminal prices) and is where
GPU memory bandwidth really shines.

In [ ]:
#@title CPU barrier option (full paths)
def barrier_call_cpu(n_paths, n_steps, S0, K, r, sigma, dt, barrier):
    z = np.random.standard_normal((n_steps, n_paths))
    paths = np.zeros((n_steps + 1, n_paths))
    paths[0] = S0
    for t in range(1, n_steps + 1):
        paths[t] = paths[t - 1] * np.exp(
            (r - 0.5 * sigma ** 2) * dt
            + sigma * np.sqrt(dt) * z[t - 1]
        )
    knocked_out = np.any(paths > barrier, axis=0)
    payoff = np.where(knocked_out, 0.0,
                      np.maximum(paths[-1] - K, 0.0))
    return np.exp(-r * T) * payoff.mean()

barrier = 120.0
price_b_cpu, t_cpu_barrier = bench(
    "CPU Barrier call",
    barrier_call_cpu,
    n_paths, n_steps, S0, K, r, sigma, dt, barrier,
)
print(f"Price: {price_b_cpu:.4f}")

In [ ]:
#@title GPU barrier option (full paths)
def barrier_call_gpu(n_paths, n_steps, S0, K, r, sigma, dt,
                     barrier, device):
    torch.manual_seed(seed)
    z = torch.randn((n_steps, n_paths), device=device)
    paths = torch.zeros((n_steps + 1, n_paths), device=device)
    paths[0] = S0
    for t in range(1, n_steps + 1):
        paths[t] = paths[t - 1] * torch.exp(
            (r - 0.5 * sigma ** 2) * dt
            + sigma * np.sqrt(dt) * z[t - 1]
        )
    knocked_out = (paths > barrier).any(dim=0)
    payoff = torch.where(
        knocked_out,
        torch.tensor(0.0, device=device),
        torch.maximum(paths[-1] - K,
                      torch.tensor(0.0, device=device)),
    )
    return (torch.exp(torch.tensor(-r * T, device=device))
            * payoff.mean()).item()

price_b_gpu, t_gpu_barrier = bench(
    "GPU Barrier call",
    barrier_call_gpu,
    n_paths, n_steps, S0, K, r, sigma, dt,
    barrier, device,
)
torch.cuda.synchronize() if device == "cuda" else None
print(f"Price: {price_b_gpu:.4f}")

speedup_barrier = t_cpu_barrier / t_gpu_barrier
print(f"\n>>> Barrier call speed-up: {speedup_barrier:.1f}×")

### Greeks: Delta via Finite Differences

We bump the initial spot by a small `dS` and re-run the simulation
to estimate Delta = (price_up - price_down) / (2 * dS).
On the GPU this is a single extra kernel launch.

In [ ]:
#@title GPU Delta estimation
dS = 1.0

S_T_up = gbm_gpu(n_paths, n_steps, S0 + dS, r, sigma, dt, device)
price_up = euro_call_gpu(S_T_up, K, r, T)

S_T_down = gbm_gpu(n_paths, n_steps, S0 - dS, r, sigma, dt, device)
price_down = euro_call_gpu(S_T_down, K, r, T)

delta_gpu = (price_up - price_down) / (2 * dS)
torch.cuda.synchronize() if device == "cuda" else None

print(f"Price at S0={S0}: {price_gpu:.4f}")
print(f"Price at S0={S0 + dS}: {price_up:.4f}")
print(f"Price at S0={S0 - dS}: {price_down:.4f}")
print(f"Delta: {delta_gpu:.4f}")

### Summary: Speed-Up Comparison

Even a modest free T4 GPU delivers 10–100× speed-ups on Monte Carlo
workloads — turning minutes into seconds for millions of paths.

In [ ]:
#@title Build and display results
import pandas as pd

results = pd.DataFrame({
    "Workload": [
        "GBM terminal (2M paths)",
        "Barrier call (2M paths, 252 steps)",
    ],
    "CPU time (s)": [
        round(t_cpu_gbm, 2),
        round(t_cpu_barrier, 2),
    ],
    "GPU time (s)": [
        round(t_gpu_gbm, 2),
        round(t_gpu_barrier, 2),
    ],
})
results["Speed-up"] = (
    results["CPU time (s)"] / results["GPU time (s)"]
)
results["Speed-up"] = (
    results["Speed-up"].round(1).astype(str) + "×"
)

print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(results))
width = 0.35
ax.bar(
    x - width / 2,
    results["CPU time (s)"],
    width,
    label="CPU",
    color="#e74c3c",
)
ax.bar(
    x + width / 2,
    results["GPU time (s)"],
    width,
    label="GPU",
    color="#2ecc71",
)
ax.set_ylabel("Wall time (seconds, log scale)")
ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels(
    results["Workload"],
    rotation=15,
    ha="right",
)
ax.legend()
ax.set_title("CPU vs GPU wall time")
plt.tight_layout()
plt.show()

### Optional: Historical Volatility from SPY

Use `yfinance` to download SPY closing prices and estimate annualized
volatility. This is a small public-data add-on that does not affect the
synthetic core of the notebook.

In [ ]:
#@title Download SPY and estimate volatility
# Uncomment the next two lines to run this section:
# !pip -q install yfinance
# import yfinance as yf
# data = yf.download("SPY", start="2020-01-01", end="2025-01-01")
# returns = np.log(data["Close"] / data["Close"].shift(1)).dropna()
# sigma_spy = returns.std() * np.sqrt(252)
# print(f"SPY annualized volatility: {sigma_spy:.4f}")

### Exercises

1. **More paths**: Increase `n_paths` to 10 M. Does the speed-up
   grow or shrink?
2. **American-style**: Use the full GPU path matrix to price a
   Bermudan option with early exercise at every 50th step.
3. **Other Greeks**: Estimate Vega and Theta via finite differences
   on the GPU.
4. **Basket option**: Simulate two correlated assets and price a
   call on their average.
5. **Real data**: Uncomment the `yfinance` section above and repeat
   the European call pricing using the estimated SPY volatility.

<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">
